In [1]:
# CELL 1: Environment, paths, and load Stage 2 outputs
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import util

EMB_DIR = Path("../data/embeddings")
OUTPUT_DIR = Path("../data/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sample = pd.read_csv(EMB_DIR / "cicids_sample.csv", index_col="sample_id", low_memory=False)
embeddings = np.load(EMB_DIR / "cicids_embeddings.npy")
assert len(sample) == len(embeddings), "Embedding/CSV length mismatch"
print(f"Loaded {len(sample)} samples | Shape: {embeddings.shape}")

Loaded 10684 samples | Shape: (10684, 384)


In [3]:
# CELL 2: Community detection (Course method)
print("Running community detection...")
communities = util.community_detection(
    embeddings,
    min_community_size=3,
    threshold=0.75,
)
print(f"Found {len(communities)} semantic communities")

Running community detection...
Found 18 semantic communities


In [4]:
# CELL 3: Map communities to DataFrame and save assignments
community_assignments = np.full(len(sample), -1, dtype=int)
for idx, comm_indices in enumerate(communities):
    for sample_idx in comm_indices:
        community_assignments[sample_idx] = idx

valid_mask = community_assignments != -1
community_df = sample[valid_mask].copy()
community_df["community_id"] = community_assignments[valid_mask]
community_df.to_csv(OUTPUT_DIR / "community_assignments.csv", index=False)
print(f"Assigned {len(community_df)} alerts to {community_df['community_id'].nunique()} communities")

Assigned 10683 alerts to 18 communities


In [5]:
# CELL 4: LLM initialization and grammar constraint
from llama_cpp import Llama, LlamaGrammar

MODEL_PATH = "../models/Mistral-Nemo-Instruct-2407-Q5_K_M.gguf"
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096,
    n_gpu_layers=12,      # Partial offload safe for 4GB VRAM
    n_threads=8,
    n_batch=256,
    verbose=False,
    seed=42,
    temperature=0,
    repeat_penalty=1.15
)

TRIPLE_SCHEMA = {
    "type": "object",
    "properties": {
        "triples": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "subject": {"type": "string", "enum": ["network_flow"]},
                    "relation": {"type": "string", "enum": ["targets_service", "shows_behavior", "has_duration", "indicates_activity", "communicates_via"]},
                    "target": {"type": "string", "enum": [
                        "ftp_service", "ssh_service", "http_service", "https_service", "dns_service", "unknown_service",
                        "very_short_connection", "long_duration", "high_volume_traffic", "incomplete_handshake",
                        "port_scanning_activity", "brute_force_login_activity", "denial_of_service_behavior",
                        "web_exploitation_activity", "command_and_control_behavior", "data_exfiltration_pattern"
                    ]}
                },
                "required": ["subject", "relation", "target"]
            }
        }
    },
    "required": ["triples"]
}
grammar = LlamaGrammar.from_json_schema(json.dumps(TRIPLE_SCHEMA))
print("LLM & Grammar initialized")

llama_context: n_ctx_seq (4096) < n_ctx_train (1024000) -- the full capacity of the model will not be utilized


LLM & Grammar initialized


In [6]:
# CELL 5: Extraction with explicit mapping and validation
ALLOWED_MAPPINGS = {
    "indicates_activity": ["port_scanning_activity", "brute_force_login_activity", "denial_of_service_behavior", "web_exploitation_activity", "command_and_control_behavior", "data_exfiltration_pattern"],
    "targets_service": ["ftp_service", "ssh_service", "http_service", "https_service", "dns_service", "unknown_service"],
    "has_duration": ["very_short_connection", "long_duration"],
    "shows_behavior": ["high_volume_traffic", "incomplete_handshake", "observed_traffic_pattern"],
    "communicates_via": ["tcp", "udp", "http", "https"]
}

def extract_triples_validated(text_block, cid):
    prompt = f"""[INST] Extract exactly 4 semantic triples summarizing this alert community.
Return valid JSON matching the schema.
STRICT PAIRING RULES:
- "indicates_activity" -> port_scanning_activity, brute_force_login_activity, denial_of_service_behavior, web_exploitation_activity, command_and_control_behavior, data_exfiltration_pattern
- "targets_service" -> ftp_service, ssh_service, http_service, https_service, dns_service, unknown_service
- "has_duration" -> very_short_connection, long_duration
- "shows_behavior" -> high_volume_traffic, incomplete_handshake, observed_traffic_pattern
ALERTS:
{text_block}
Return ONLY valid JSON. [/INST]"""

    out = llm(prompt, max_tokens=400, temperature=0, seed=42, grammar=grammar, repeat_penalty=1.15, stop=["[/INST]"])
    raw = out["choices"][0]["text"].strip()

    try:
        triples = json.loads(raw).get("triples", [])
    except Exception:
        triples = []

    # Validate and correct relation-target pairs deterministically
    validated = []
    for t in triples:
        rel = str(t.get("relation", "")).lower()
        tgt = str(t.get("target", "")).lower()
        allowed = ALLOWED_MAPPINGS.get(rel, [])
        if tgt not in allowed:
            tgt = allowed[0] if allowed else "observed_traffic_pattern"
        validated.append({"subject": "network_flow", "relation": rel, "target": tgt})

    # Ensure exactly 4 triples
    default = {"subject": "network_flow", "relation": "shows_behavior", "target": "observed_traffic_pattern"}
    validated = (validated + [default] * 4)[:4]
    return validated

In [7]:
# CELL 6: Warmup and run extraction loop
print("Warming up LLM context...")
_ = llm("[INST] Warmup.[/INST]", max_tokens=5, temperature=0, seed=42, grammar=grammar)
print("Warmup complete.")

community_triples = {}
for cid, group in community_df.groupby("community_id"):
    texts = group["alert_text"].dropna().head(6).tolist()
    block = "\n".join([f"- {t}" for t in texts])
    community_triples[str(int(cid))] = extract_triples_validated(block, int(cid))

print(f"Extracted triples for {len(community_triples)} communities")

Warming up LLM context...
Warmup complete.
Extracted triples for 18 communities


In [8]:
# CELL 7: Save outputs, compute metrics, and inspect 3 communities
valid_count = sum(1 for t in community_triples.values() for triple in t if all(k in triple for k in ["subject","relation","target"]))
total_count = sum(len(t) for t in community_triples.values())

triple_metrics = {
    "n_communities": len(community_triples),
    "mean_triples_per_community": float(np.mean([len(v) for v in community_triples.values()])),
    "valid_ratio": round(valid_count / max(total_count, 1), 4),
    "failed_communities": 0
}

with open(OUTPUT_DIR / "triple_metrics.json", "w") as f:
    json.dump(triple_metrics, f, indent=2)
with open(OUTPUT_DIR / "community_triples.json", "w", encoding="utf-8") as f:
    json.dump(community_triples, f, indent=2)

print("Clustering & Triple metrics:", triple_metrics)

print("\n=== INSPECTION: FIRST 3 COMMUNITIES ===")
for cid in list(community_triples.keys())[:3]:
    group = community_df[community_df["community_id"] == int(cid)]
    print(f"Community {cid} | Label: {group['Label'].mode().iloc[0]} | Tactic: {group['attck_tactic'].mode().iloc[0]}")
    for t in community_triples[cid]:
        print(f"  network_flow --[{t['relation']}]--> {t['target']}")
    print("-" * 50)

Clustering & Triple metrics: {'n_communities': 18, 'mean_triples_per_community': 4.0, 'valid_ratio': 1.0, 'failed_communities': 0}

=== INSPECTION: FIRST 3 COMMUNITIES ===
Community 0 | Label: PortScan | Tactic: Discovery
  network_flow --[indicates_activity]--> port_scanning_activity
  network_flow --[targets_service]--> http_service
  network_flow --[shows_behavior]--> high_volume_traffic
  network_flow --[has_duration]--> long_duration
--------------------------------------------------
Community 1 | Label: BENIGN | Tactic: Benign
  network_flow --[targets_service]--> dns_service
  network_flow --[indicates_activity]--> port_scanning_activity
  network_flow --[has_duration]--> very_short_connection
  network_flow --[shows_behavior]--> high_volume_traffic
--------------------------------------------------
Community 2 | Label: SSH Patator | Tactic: Credential Access
  network_flow --[targets_service]--> ftp_service
  network_flow --[has_duration]--> very_short_connection
  network_flow